# Error404 · OrientaFP Canarias · Baseline TF-IDF
Sprint 14. Actualizamos nuestro punto de partida con el corpus definitivo. Mantenemos TF-IDF y búsqueda top-1 sin generación, con la misma configuración y las 12 preguntas de desarrollo originales.

## 1. Partición y métrica fijadas previamente
Indexamos todos los documentos públicos: no entrenamos un clasificador ni dividimos el corpus en entrenamiento y test. Separamos las preguntas de desarrollo del test final, que no cargamos ni ejecutamos. Conservamos la semilla 42 de la selección original (la recuperación es determinista). Medimos Hit@1: proporción de preguntas cuyo primer documento tiene el identificador de evidencia esperado y similitud positiva. Nuestro objetivo final de 36/40 respuestas correctas es una métrica distinta.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
SEMILLA = 42

def localizar(nombre):
    for ruta in [Path(nombre), Path('corpus_definitivo') / nombre, Path('..') / nombre, Path('..') / 'corpus_definitivo' / nombre]:
        if ruta.is_file():
            return ruta
    return None

nombres = ['corpus_fp_canarias_ampliado.csv', 'preguntas_desarrollo.csv']
if any(localizar(f) is None for f in nombres):
    from google.colab import files as colab_files
    print('Subimos los dos CSV indicados, no el test final.')
    colab_files.upload()
rutas = [localizar(f) for f in nombres]
assert all(r is not None for r in rutas), 'Faltan archivos de entrada'
huella = hashlib.sha256(rutas[0].read_bytes()).hexdigest()
assert huella == 'be86c43d3f3604ff07fafebb13acba86a2899f19af6cf1eea4d0099445e9765f', 'El corpus no coincide con la versión definitiva'
docs = pd.read_csv(rutas[0], dtype=str).fillna('')
qs = pd.read_csv(rutas[1], dtype=str).fillna('')
assert len(docs) == 2914 and docs.id_documento.is_unique
assert docs.texto.str.strip().ne('').all()
assert len(qs) == 12 and qs.id.is_unique
assert qs.evidencia.isin(docs.id_documento).all()
print('SHA-256 corpus:', huella)
print(len(docs), 'documentos;', len(qs), 'preguntas de desarrollo')
display(docs.tipo_documento.value_counts())

SHA-256 corpus: be86c43d3f3604ff07fafebb13acba86a2899f19af6cf1eea4d0099445e9765f
2914 documentos; 12 preguntas de desarrollo


tipo_documento
oferta            2751
informacion_fp     163
Name: count, dtype: int64

## 2. Recuperación TF-IDF
Convertimos los textos y preguntas en vectores TF-IDF y calculamos su similitud coseno. Conservamos los parámetros originales; no ajustamos el modelo a los resultados. Usamos el identificador estable del documento, válido también para las fichas sin código de centro.

In [2]:
vectorizador = TfidfVectorizer(lowercase=True, strip_accents='unicode')
X = vectorizador.fit_transform(docs['texto'])
Q = vectorizador.transform(qs['pregunta'])
scores = cosine_similarity(Q, X)
indices = scores.argmax(axis=1)
resultados = qs.copy()
resultados['recuperado'] = docs.iloc[indices].id_documento.to_numpy()
resultados['tipo_recuperado'] = docs.iloc[indices].tipo_documento.to_numpy()
resultados['similitud'] = scores.max(axis=1)
resultados['acierto'] = (resultados.recuperado == resultados.evidencia) & (resultados.similitud > 0)
resultados['texto_recuperado'] = docs.iloc[indices].texto.to_numpy()
display(resultados[['id','pregunta','evidencia','recuperado','tipo_recuperado','similitud','acierto']])
print(f'Nuestro baseline tiene Hit@1 de {resultados.acierto.mean():.2%} ({resultados.acierto.sum()}/{len(resultados)}) en desarrollo.')

,id,pregunta,evidencia,recuperado,tipo_recuperado,similitud,acierto
0,P01,¿Dónde se imparte 1º CFGM Administración y Ges...,38011133/8602/2025/26,38011133/8602/2025/26,oferta,0.565151,True
1,P02,¿Dónde se imparte 2º CFGS Hostelería y turismo...,35015875/4835/2025/26,35015875/4835/2025/26,oferta,0.624047,True
2,P03,¿Dónde se imparte 1º CFGS Semi. Comercio y Mar...,38003963/8733/2025/26,38003963/8733/2025/26,oferta,0.628541,True
3,P04,¿Dónde se imparte 1º CFGM Informática y Comuni...,38016763/8596/2025/26,38016763/8596/2025/26,oferta,0.629368,True
4,P05,¿Dónde se imparte 2º CFGM Agraria - Aprovecham...,38006174/7086/2025/26,38006174/7086/2025/26,oferta,0.626458,True
5,P06,¿Dónde se imparte 2º CFGM Dist. Servicios Soci...,35014664/6097/2025/26,35014664/8340/2025/26,oferta,0.685987,False
6,P07,¿Dónde se imparte 1º CFGM Comercio y Marketing...,35010476/8611/2025/26,35010476/8611/2025/26,oferta,0.513638,True
7,P08,¿Dónde se imparte 1º CFGS Sanidad - Laboratori...,38016519/8686/2025/26,38016519/8686/2025/26,oferta,0.577223,True
8,P09,¿Dónde se imparte 1º CFGS Semi. Servicios Soci...,35014585/8718/2025/26,35014585/8718/2025/26,oferta,0.577314,True
9,P10,¿Dónde se imparte 1º CFGS Comercio y Marketing...,38002791/8669/2025/26,38002791/8669/2025/26,oferta,0.640722,True


Nuestro baseline tiene Hit@1 de 91.67% (11/12) en desarrollo.


## 3. Analizamos los errores sin modificar el baseline

In [3]:
for _, fila in resultados.loc[~resultados.acierto].iterrows():
    print(fila.id, fila.pregunta)
    print('Esperado:', docs.loc[docs.id_documento == fila.evidencia, 'texto'].iloc[0])
    print('Recuperado:', fila.texto_recuperado)
    print('Similitud:', fila.similitud)
display(resultados.tipo_recuperado.value_counts())

P06 ¿Dónde se imparte 2º CFGM Dist. Servicios Socioculturales y a la Comunidad - Atención a Personas en Situación de Dependencia (LOE) en CIFP VILLA DE AGÜIMES? Indica municipio e isla en 2025/26.
Esperado: CodigoCentro: 35014664 | CodigoEstudio: 6097 | DenominacionCentro: CIFP VILLA DE AGÜIMES | Isla: GRAN CANARIA | Municipio: AGÜIMES | Nivel: Ciclo Formativo de Grado Medio a Distancia | Familia: Servicios Socioculturales y a la Comunidad (LOE) | Estudio: 2º CFGM Dist. Servicios Socioculturales y a la Comunidad - Atención a Personas en Situación de Dependencia (LOE) | Modalidad: A distancia | CursoEscolar: 2025/26
Recuperado: CodigoCentro: 35014664 | CodigoEstudio: 8340 | DenominacionCentro: CIFP VILLA DE AGÜIMES | Isla: GRAN CANARIA | Municipio: AGÜIMES | Nivel: Ciclo Formativo de Grado Medio | Familia: Servicios Socioculturales y a la Comunidad (LOE) | Estudio: 2º CFGM Servicios Socioculturales y a la Comunidad - Atención a Personas en Situación de Dependencia (LOE) | Modalidad: Pre

tipo_recuperado
oferta    12
Name: count, dtype: int64

## 4. Nuestras conclusiones
Obtenemos 11/12 aciertos (91,67 %), igual que en el baseline anterior. El error P06 recupera otro código de estudio del mismo centro. No cambiamos los parámetros para corregirlo. Las 12 preguntas recuperan documentos de oferta: no podemos afirmar que el sistema recupere bien los requisitos o las salidas profesionales con esta evaluación.

Hit@1 mide recuperación exacta, no veracidad de respuestas generadas. No tenemos un umbral de abstención calibrado ni evaluación de preguntas multidocumento, sin respuesta o adversarias. Para evaluar las fichas necesitaremos preguntas de desarrollo independientes del test. Las fichas son más largas que los documentos de oferta y podrían influir en la recuperación. Conservamos las limitaciones de calidad y reutilización del plan de datos, incluida la revisión humana pendiente. No ejecutamos el test final ni modificamos los CSV.